# Feature Aggregation

This notebook merges all extracted feature files into a single dataset for machine learning.

**Input:**
- Individual feature pickle files from `features_mlcrowd/`

**Output:**
- `features_master.pkl`: Consolidated dataframe with all features


In [5]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

# Set display options
pd.set_option('display.max_columns', None)

# =============================================================================
# DIRECTORY PATHS
# =============================================================================
# Define the base data directory (adjust if needed based on your local setup)
DATA_DIR = Path(r"c:\Users\skazempour\Documents\StockTwits\dataset\v1\data\csv")
FEATURES_DIR = DATA_DIR / "features_mlcrowd"
OUTPUT_FILE = FEATURES_DIR / "features_master.pkl"

print(f"Features Directory: {FEATURES_DIR}")
print(f"Output File: {OUTPUT_FILE}")

# List available feature files
if FEATURES_DIR.exists():
    feature_files = sorted([f for f in FEATURES_DIR.glob("*.pkl") if "master" not in f.name])
    print(f"\nFound {len(feature_files)} feature files:")
    for f in feature_files:
        print(f" - {f.name}")
else:
    print(f"Directory not found: {FEATURES_DIR}")


Features Directory: c:\Users\skazempour\Documents\StockTwits\dataset\v1\data\csv\features_mlcrowd
Output File: c:\Users\skazempour\Documents\StockTwits\dataset\v1\data\csv\features_mlcrowd\features_master.pkl

Found 3 feature files:
 - features_01_basic_sentiment.pkl
 - features_02_volume_attention.pkl
 - features_03_abnormal_sentiment.pkl


In [6]:
# Load and merge all feature files
if not feature_files:
    raise FileNotFoundError("No feature files found to merge.")

# Start with the first file
print(f"Loading base file: {feature_files[0].name}")
df_master = pd.read_pickle(feature_files[0])
print(f"  Shape: {df_master.shape}")

# Merge remaining files
for file_path in feature_files[1:]:
    print(f"\nMerging: {file_path.name}")
    df_next = pd.read_pickle(file_path)
    print(f"  Shape: {df_next.shape}")
    
    # Check for duplicate columns (excluding keys)
    common_cols = [c for c in df_next.columns if c in df_master.columns and c not in ['symbol', 'date']]
    if common_cols:
        print(f"  Warning: Duplicate columns found (keeping original): {common_cols}")
        df_next = df_next.drop(columns=common_cols)
    
    # Merge on symbol and date
    # Using outer merge to keep all data
    df_master = pd.merge(df_master, df_next, on=['symbol', 'date'], how='outer')

print(f"\n{'='*40}")
print(f"Merge Complete")
print(f"{'='*40}")
print(f"Final Shape: {df_master.shape}")
print(f"Unique Symbols: {df_master['symbol'].nunique()}")
print(f"Date Range: {df_master['date'].min()} to {df_master['date'].max()}")

Loading base file: features_01_basic_sentiment.pkl
  Shape: (3505817, 13)

Merging: features_02_volume_attention.pkl
  Shape: (14193826, 22)

Merging: features_03_abnormal_sentiment.pkl
  Shape: (28206145, 18)

Merge Complete
Final Shape: (28206145, 38)
Unique Symbols: 8245
Date Range: 2010-06-02 00:00:00 to 2024-01-03 00:00:00


In [7]:
# Inspect the merged dataframe
print("Columns in master dataframe:")
print(list(df_master.columns))

# Check for nulls
print("\nMissing values per column:")
print(df_master.isnull().sum()[df_master.isnull().sum() > 0])

# Save to pickle
print(f"\nSaving master feature file to: {OUTPUT_FILE}")
df_master.to_pickle(OUTPUT_FILE)
print("✓ Saved successfully!")


Columns in master dataframe:
['symbol', 'date', 'n_bullish', 'n_bearish', 'total_labeled', 'bullish_ratio', 'bearish_ratio', 'net_sentiment', 'extreme_bullish_80', 'extreme_bullish_90', 'extreme_bearish_80', 'extreme_bearish_90', 'disagreement_index', 'raw_volume', 'log_volume', 'unique_user_count', 'volume_diff', 'log_volume_change', 'abn_volume_5d', 'abn_attention_std_5d', 'attention_surge_5d', 'abn_volume_21d', 'abn_attention_std_21d', 'attention_surge_21d', 'abn_volume_63d', 'abn_attention_std_63d', 'attention_surge_63d', 'abn_volume_250d', 'abn_attention_std_250d', 'attention_surge_250d', 'silence_gap_hours', 'relative_volume', 'attention_hhi', 'abnormal_sentiment_1d', 'abnormal_sentiment_5d', 'abnormal_sentiment_21d', 'abnormal_sentiment_63d', 'abnormal_sentiment_250d']

Missing values per column:
n_bullish                 24700328
n_bearish                 24700328
total_labeled             24700328
bullish_ratio             24700328
bearish_ratio             24700328
net_sentim

## Next Steps
Now that the features are merged, you can proceed to:
1.  **Data Cleaning:** Handle any missing values identified in the inspection step.
2.  **Train/Test Split:** Split the data for model validation.
3.  **Model Training:** Use the `features_master.pkl` file in your prediction notebooks.
